In [2]:
 #Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# res='1km'
# Np_str='1e6'

# dx = 1 km; Np = 1M; Nt = 1 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_1min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_1min.nc') #***
res='1km'
Np_str='1e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [3]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [5]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 0, end_job = 16667


In [ ]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min.h5'
with h5py.File(in_file, 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# # #Making Time Matrix
# Nt=len(data['time'])
# T = np.broadcast_to(np.arange(Nt)[:, None], A_c.shape)  # shape: (Nt, p)

In [17]:
###########################################################################################################################################################################

In [4]:
def extend_idxs(f,case):
    out=np.sort(np.add.outer(f, np.arange(case)).ravel())

    # #OLD METHOD (SLOW)
    # if np.any(f)==True:
    #     out=np.sort(np.concatenate([np.arange(idx, idx + case-1+1) for idx in f]))
    # else: 
    #     out=f
    return out

def find_sandwiched_patterns(changes, case):
    arr=changes
    
    window_size = case + 1  # e.g., for case=2, window_size = 3
    # The interior zeros count is (window_size - 2) which is case - 1
    pattern1 = np.array([-1] + [0]*(case - 1) + [1])
    pattern2 = np.array([1] + [0]*(case - 1) + [-1])
    
    # Manually construct sliding windows
    windows = np.array([arr[i:i + window_size] for i in range(len(arr) - window_size + 1)])
    # print("Sliding windows:\n", windows) #TESTING
    
    #THE ALGORITHM
    turb_d=[]
    turb_e=[]
    count=0;max_iter=len(data['time']);
    while np.any(((windows == pattern1) | (windows == pattern2)).all(axis=1)):
        count+=1; 
        if count>=max_iter: 
            print(count)
            break
        
        next_ind = np.where(((windows == pattern1) | (windows == pattern2)).all(axis=1))[0][0]
        
        if (windows[next_ind] == pattern1).all():
            turb_d.append(next_ind)
        elif (windows[next_ind] == pattern2).all(): 
            turb_e.append(next_ind) #append to list
    
        windows[0:next_ind+(case)+1,:] = 0 #removes from windows
    
    turb_d=np.array(turb_d,dtype=int); turb_e=np.array(turb_e,dtype=int)

    #EXTEND REST OF INDEXES TO PROCESS
    turb_d=extend_idxs(turb_d,case=case)
    turb_e=extend_idxs(turb_e,case=case)
    return turb_d,turb_e

In [343]:
# # TESTING
# # changes = np.array([0,0,0,-1,1,0,0,-1,0,0,0,1,-1,0,0])
# [a,b] = find_sandwiched_patterns(changes, case=1) #<=1 in a row timesteps are removed
# print("Case matches at indices:", a,b)

# changes = np.array([0,0,0,-1,0,1,0,0,-1,0,0,1,0,-1,0,0])
# [a,b] = find_sandwiched_patterns(changes, case=2) #<=2 in a row timesteps are removed
# print("Case matches at indices:", a,b)

# changes = np.array([0,0,0,-1,0,0,1,0,0,0,0,1,0,0,-1,0,0])
# [a,b] = find_sandwiched_patterns(changes, case=3) #<=3 in a row timesteps are removed
# print("Case matches at indices:", a,b)

In [2]:
###### (amount of time outside of cloud to count as entrainment)
mins_thresh=5 #5 mins
######

def get_changes(B):
    changes = np.diff(np.concatenate(([B[0]], B)))  # Add 0s to detect edges
    return changes
def PreProcessing(p,updraft_type):

    if updraft_type=='general':
        A=A_g
    elif updraft_type=='cloudy':
        A=A_c
    B = A[:,p]

    # Find the changes in the array
    changes=get_changes(B)
    # print(f'B = {B}'); print(f'changes = {changes}') #TESTING

    #Determining the Case Number
    t_per_mins=1/((data['time'][1]-data['time'][0])/1e9/60).item() #timesteps per minute (<=1)
    case=int(t_per_mins*mins_thresh)
    # case=2 #TESTING
    
    if case>1:
        for case_ind in np.arange(case,0,-1): 
            #Calling Algorithm and Correcting Parcel Data
            [turb_d,turb_e]=find_sandwiched_patterns(changes, case=case_ind)
            B[turb_d]=1
            B[turb_e]=0     
            changes=get_changes(B)
            # print(B) #TESTING
    elif case==1:
        #Calling Algorithm and Correcting Parcel Data
        [turb_d,turb_e]=find_sandwiched_patterns(changes, case=case)
        B[turb_d]=1
        B[turb_e]=0
    return B

In [1]:
# TESTING
# Case 1
# B=np.array([1,0,1,1,0,0,1,0])
# B=np.array([1,0,1,1,0,1,0]) 
# B=np.array([1,0,1,1,0,1,0,1])

# B=np.array([1,0,1,0,1,1,1])
# B=np.array([1,0,1,0,1,0,0])
# B=np.array([0,1,0,1,0,0,0]) 
# B=np.array([0,1,0,1,0,1,1])

# Case 2
# B=np.array([1,1,1,0,0,1,1,1,0,0,0,1,1,0,0,0])
# B=np.array([1,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0]) #101 should still get removed

# # Case 3
# B=np.array([1,1,1,1,0,0,0,1,1,1,1,0,0,0,1,0,0,1])
# p=999857; out=PreProcessing(p,updraft_type='cloudy') #TESTING
# print(f'output =  {out}\n')

# #TESTING
# count_per_row = (A_c >= 1).sum(axis=0)
# where=np.where(count_per_row > 10)[0] # Find row indices where count is greater than 10
# print(where)
# p=999697; out=PreProcessing(p,updraft_type='cloudy') #TESTING
# print(f'output =  {out}\n')

In [ ]:
# #RUNNING (WITHOUT JOB_ARRAY)
# A_g_Processed=A_g.copy() 
# A_c_Processed=A_c.copy()
# print('processing parcels for general updrafts')
# for p in np.arange(len(parcel['xh'])):
#     # if p==1000:break #TESTING
#     if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
#     out=PreProcessing(p,updraft_type='general'); A_g_Processed[:,p]=out
# print('processing parcels for cloudy updrafts')
# for p in np.arange(len(parcel['xh'])):
#     # if p==1000:break #TESTING
#     if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
#     out=PreProcessing(p,updraft_type='cloudy'); A_c_Processed[:,p]=out
    

# #SAVING
# mins_thresh=5 #5 minutes
# # out_file=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}.h5'
# out_file=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}_1min.h5'
# with h5py.File(out_file, 'w') as h5file:
#     h5file.create_dataset('A_g_Processed', data=A_g_Processed)
#     h5file.create_dataset('A_c_Processed', data=A_c_Processed)
# print('done')
# #OLD ALGORITHM about 50 minutes for 1 million parcels (general+cloudy)
# #NEW ALGORITHM ESTIMATION ==> 3.53 s ± 8.47 ms per run (tested 7 runs) ==> average = (1e6*3.53/1000)/60 = 58 minutes

In [14]:
#JOB_ARRAY VERSION
########################################################################

In [ ]:
#RUNNING
A_g_Processed=A_g.copy()
A_c_Processed=A_c.copy()
print('processing parcels for general updrafts')
for p in np.arange(start_job,end_job):
    # if p==1000:break #TESTING
    if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
    out=PreProcessing(p,updraft_type='general'); A_g_Processed[:,p]=out
    
print('processing parcels for cloudy updrafts')
for p in np.arange(start_job,end_job):
    # if p==1000:break #TESTING
    if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
    out=PreProcessing(p,updraft_type='cloudy'); A_c_Processed[:,p]=out
    

#SAVING
mins_thresh=5 #5 minutes
# out_file=dir+f'Project_Algorithms/Entrainment/job_out_3/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}_{job_id}.h5'
out_file=dir+f'Project_Algorithms/Entrainment/job_out_3/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}_1min_{job_id}.h5'
with h5py.File(out_file, 'w') as h5file:
    h5file.create_dataset('A_g_Processed', data=A_g_Processed)
    h5file.create_dataset('A_c_Processed', data=A_c_Processed)
print('done')


In [ ]:
#COMBINING JOB_ARRAYS AFTER RUNNING
########################################################################

In [9]:
### SAVING
#############################################################################################################
mins_thresh=5

Nt=len(data['time']); Np=len(parcel['xh'])
A_g_Processed = np.zeros((Nt,Np),dtype=int)
A_c_Processed = np.zeros((Nt,Np),dtype=int)

# out_file=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}.h5'
out_file=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}_1min.h5'
with h5py.File(out_file, 'w') as h5file:
    h5file.create_dataset('A_g_Processed', data=A_g_Processed)
    h5file.create_dataset('A_c_Processed', data=A_c_Processed)
print('done')

print('combining')
mins_thresh=5 #5 minutes
num_jobs=60
for job_id in np.arange(1,num_jobs+1):
    print(job_id)
    [start_job,end_job]=get_job_range(job_id)
    
    # in_file=dir+f'Project_Algorithms/Entrainment/job_out_3/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}_{job_id}.h5'
    in_file=dir+f'Project_Algorithms/Entrainment/job_out_3/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}_1min_{job_id}.h5'
    
    with h5py.File(in_file, 'r') as f_in:
        A_g_Processed_job = f_in['A_g_Processed'][:, start_job:end_job]
        A_c_Processed_job = f_in['A_c_Processed'][:, start_job:end_job]
        
    # Open the output file inside the loop to write the data
    with h5py.File(out_file, 'r+') as f_out:
        f_out['A_g_Processed'][:, start_job:end_job] = A_g_Processed_job
        f_out['A_c_Processed'][:, start_job:end_job] = A_c_Processed_job
    
print('done')

done
combining
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
done
